<a href="https://colab.research.google.com/github/kawastony/Quadratic-Mechanism-Lens/blob/main/TIFA_Figures_paper1.5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# Figure 2 for TIFA Paper 1.5
# Two panels:
#   Top:    v_phi(z) — field velocity parameter
#   Bottom: m/H(z)  — mass vs Hubble rate
# Save as: figure2_paper1point5.pdf
# Run: python figure2_paper1point5.py
# ============================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'font.family':     'serif',
    'font.size':       12,
    'axes.labelsize':  13,
    'legend.fontsize': 10.5,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'text.usetex':     False,
    'figure.dpi':      150,
    'axes.linewidth':  1.2,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top':       True,
    'ytick.right':     True,
})

# ── TIFA parameters ──────────────────────────────────────────
feff     = 0.305
Omega_DE = 0.69
Omega_m  = 0.31
Omega_r  = 9.0e-5
H0       = 1.0

rho_DE_0 = 3.0 * H0**2 * Omega_DE
Lambda4  = 0.1 * rho_DE_0
phi_i    = 0.377 * np.pi * feff

def V(phi):
    return Lambda4 * (1.0 + np.cos(phi / feff))

def dV(phi):
    return -Lambda4 / feff * np.sin(phi / feff)

def d2V(phi):
    return -Lambda4 / feff**2 * np.cos(phi / feff)

# ── ODE system (same as Figure 1) ───────────────────────────
def odes(lna, y):
    phi, phidot_H = y
    a = np.exp(lna)
    rho_m  = Omega_m * a**(-3)
    rho_r  = Omega_r * a**(-4)
    Vphi   = V(phi)
    dVphi  = dV(phi)
    denom  = 3.0 - 0.5 * phidot_H**2
    if denom <= 0:
        denom = 1e-10
    H2 = (rho_m + rho_r + Vphi) / denom
    H  = np.sqrt(max(H2, 1e-30))
    phidot = phidot_H * H
    rho_phi = 0.5 * phidot**2 + Vphi
    rho_tot = rho_m + rho_r + rho_phi
    p_phi   = 0.5 * phidot**2 - Vphi
    p_tot   = rho_r / 3.0 + p_phi
    Hdot_over_H2 = -0.5 * (rho_tot + p_tot) / H2
    dphi_dlna = phidot / H
    ddotphi = -3.0 * H * phidot - dVphi
    d_phidotH_dlna = (
        ddotphi / H - phidot_H * Hdot_over_H2 * H
    ) / H
    return [dphi_dlna, d_phidotH_dlna]

# ── Integrate ────────────────────────────────────────────────
a_i   = 1.0 / 5.0
lna_i = np.log(a_i)
lna_f = 0.0

H_i_approx = np.sqrt(
    (Omega_m * a_i**(-3) + Omega_r * a_i**(-4) + V(phi_i)) / 3.0
)
phidot_i   = -dV(phi_i) / (3.0 * H_i_approx)
phidot_H_i = phidot_i / H_i_approx

sol = solve_ivp(
    odes, (lna_i, lna_f), [phi_i, phidot_H_i],
    t_eval=np.linspace(lna_i, lna_f, 600),
    method='DOP853', rtol=1e-9, atol=1e-11
)

a_arr  = np.exp(sol.t)
z_arr  = 1.0 / a_arr - 1.0
phi_a  = sol.y[0]
phiH_a = sol.y[1]

# ── Compute v_phi and m/H ────────────────────────────────────
vphi_arr = np.zeros(len(z_arr))
mH_arr   = np.zeros(len(z_arr))

for i in range(len(z_arr)):
    phi = phi_a[i]
    xH  = phiH_a[i]
    a   = a_arr[i]
    Vphi = V(phi)
    rho_m = Omega_m * a**(-3)
    rho_r = Omega_r * a**(-4)
    denom = 3.0 - 0.5 * xH**2
    H2 = (rho_m + rho_r + Vphi) / max(denom, 1e-10)
    H  = np.sqrt(max(H2, 1e-30))
    phidot = xH * H
    # v_phi = |phidot| / (H * feff)
    vphi_arr[i] = abs(phidot) / (H * feff)
    # m^2 = |V''(phi)|
    m2 = abs(d2V(phi))
    mH_arr[i] = np.sqrt(m2) / H

# Sort ascending z
idx    = np.argsort(z_arr)
z_plot = z_arr[idx]
vp     = vphi_arr[idx]
mH     = mH_arr[idx]

# ── Stability bands: +/-5% on feff ──────────────────────────
def run_model(feff_val, phi_i_val, Lambda4_val):
    def V2(phi):
        return Lambda4_val * (1.0 + np.cos(phi / feff_val))
    def dV2(phi):
        return -Lambda4_val / feff_val * np.sin(phi / feff_val)
    def d2V2(phi):
        return -Lambda4_val / feff_val**2 * np.cos(phi / feff_val)
    def odes2(lna, y):
        phi, phidot_H = y
        a = np.exp(lna)
        rho_m  = Omega_m * a**(-3)
        rho_r  = Omega_r * a**(-4)
        Vphi   = V2(phi)
        dVphi  = dV2(phi)
        denom  = 3.0 - 0.5 * phidot_H**2
        if denom <= 0:
            denom = 1e-10
        H2 = (rho_m + rho_r + Vphi) / denom
        H  = np.sqrt(max(H2, 1e-30))
        phidot  = phidot_H * H
        rho_phi = 0.5 * phidot**2 + Vphi
        rho_tot = rho_m + rho_r + rho_phi
        p_phi   = 0.5 * phidot**2 - Vphi
        p_tot   = rho_r / 3.0 + p_phi
        Hdot_H2 = -0.5 * (rho_tot + p_tot) / H2
        dphi_dlna = phidot / H
        ddotphi = -3.0 * H * phidot - dVphi
        d_phiH_dlna = (ddotphi / H - phidot_H * Hdot_H2 * H) / H
        return [dphi_dlna, d_phiH_dlna]

    H_i = np.sqrt(
        (Omega_m * a_i**(-3) + Omega_r * a_i**(-4) + V2(phi_i_val)) / 3.0
    )
    pd_i = -dV2(phi_i_val) / (3.0 * H_i)
    pH_i = pd_i / H_i
    s = solve_ivp(
        odes2, (lna_i, lna_f), [phi_i_val, pH_i],
        t_eval=np.linspace(lna_i, lna_f, 600),
        method='DOP853', rtol=1e-9, atol=1e-11
    )
    a2 = np.exp(s.t)
    z2 = 1.0 / a2 - 1.0
    vp2 = np.zeros(len(z2))
    mH2 = np.zeros(len(z2))
    for i in range(len(z2)):
        ph  = s.y[0][i]
        xH  = s.y[1][i]
        a_  = a2[i]
        Vp  = V2(ph)
        rm  = Omega_m * a_**(-3)
        rr  = Omega_r * a_**(-4)
        dn  = 3.0 - 0.5 * xH**2
        H2_ = (rm + rr + Vp) / max(dn, 1e-10)
        H_  = np.sqrt(max(H2_, 1e-30))
        pd_ = xH * H_
        vp2[i] = abs(pd_) / (H_ * feff_val)
        m2_ = abs(d2V2(ph))
        mH2[i] = np.sqrt(m2_) / H_
    idx2 = np.argsort(z2)
    return z2[idx2], vp2[idx2], mH2[idx2]

z_hi, vp_hi, mH_hi = run_model(
    feff * 1.05, phi_i * 1.05, Lambda4 * 1.05
)
z_lo, vp_lo, mH_lo = run_model(
    feff * 0.95, phi_i * 0.95, Lambda4 * 0.95
)

# ── Two-panel figure ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(7.5, 8.5), sharex=True
)
fig.subplots_adjust(hspace=0.08)

# ── Top panel: v_phi(z) ──────────────────────────────────────
ax1.fill_between(
    z_hi, vp_lo, vp_hi,
    alpha=0.25, color='#E74C3C',
    label=r'$\pm5\%$ parameter variation'
)
ax1.plot(
    z_plot, vp,
    color='#C0392B', linewidth=2.5,
    label=r'TIFA baseline'
)
ax1.axhline(
    1.0, color='gray', linewidth=1.2,
    linestyle=':', label='Slow-roll boundary'
)
ax1.set_ylabel(
    r'Field velocity $v_\phi(z) = |\dot\phi|/(H f_{\rm eff})$',
    labelpad=8
)
ax1.set_ylim(0, 0.45)
ax1.set_xlim(0, 4)
ax1.legend(loc='upper right', framealpha=0.9)
ax1.grid(True, alpha=0.25, linestyle='--')
ax1.set_title(
    r'TIFA Internal Consistency: $v_\phi(z)$ and $m/H(z)$',
    pad=10
)
ax1.annotate(
    r'Slow roll: $v_\phi \ll 1$ throughout',
    xy=(0.04, 0.85), xycoords='axes fraction',
    fontsize=10.5,
    bbox=dict(boxstyle='round,pad=0.3',
              facecolor='white', alpha=0.85)
)

# ── Bottom panel: m/H(z) ─────────────────────────────────────
ax2.fill_between(
    z_hi, mH_lo, mH_hi,
    alpha=0.25, color='#2E86C1',
    label=r'$\pm5\%$ parameter variation'
)
ax2.plot(
    z_plot, mH,
    color='#1A5276', linewidth=2.5,
    label=r'TIFA baseline'
)
ax2.axhline(
    1.0, color='gray', linewidth=1.2,
    linestyle='--', label=r'$m = H$ (marginal slow roll)'
)
ax2.set_xlabel(r'Redshift $z$', labelpad=8)
ax2.set_ylabel(
    r'Mass-to-Hubble ratio $m/H(z)$',
    labelpad=8
)
ax2.set_ylim(0, 1.6)
ax2.legend(loc='upper right', framealpha=0.9)
ax2.grid(True, alpha=0.25, linestyle='--')
ax2.annotate(
    r'$m/H_0 \simeq 0.91$ at $z=0$',
    xy=(0.04, 0.12), xycoords='axes fraction',
    fontsize=10.5,
    bbox=dict(boxstyle='round,pad=0.3',
              facecolor='white', alpha=0.85)
)

# Panel labels
ax1.text(
    0.97, 0.06, r'(a)', transform=ax1.transAxes,
    fontsize=12, ha='right', va='bottom',
    fontweight='bold'
)
ax2.text(
    0.97, 0.06, r'(b)', transform=ax2.transAxes,
    fontsize=12, ha='right', va='bottom',
    fontweight='bold'
)

plt.savefig(
    'figure2_paper1point5.pdf', bbox_inches='tight'
)
plt.savefig(
    'figure2_paper1point5.png',
    bbox_inches='tight', dpi=200
)
print("Saved: figure2_paper1point5.pdf and .png")

Saved: figure2_paper1point5.pdf and .png
